In [54]:
import os
import numpy as np
import pandas as pd
import pydhn
from pydhn.fluids import Fluid
from pydhn.soils import Soil
from pydhn.solving import SimpleStep

# Parameters

In [55]:
t_supply_producer = 70 # °C
mdot_consumers = 553 / 3600 #kg/s
delta_t_consumers = - 30 # K

rho_pipe = 940 # kg/m3
lambda_pipe = 0.35 # W/kg/k
cp_pipe = 2000 # J/kg/K
roughness_pipe = 0.007 # mm

rho_ins = 50 # kg/m3
lambda_ins = 0.026 # W/kg/k
cp_ins = 1500 # J/kg/K

# Network creation

In [56]:

def create_net():
    # Load the files with the DESTEST data
    node_data = pd.read_csv("InputData/nodes_data.csv", sep=';')
    pipe_data = pd.read_csv("InputData/pipes_data.csv", sep=';')

    # Create network
    net = pydhn.Network()

    # Add nodes
    for i in node_data.index:
        row = node_data.loc[i]
        name = row["node_id"]
        name_s = name + "_s"
        name_r = name + "_r"
        X = row["x"]
        Y = row["y"]
        net.add_node(name=name_s, x=X, y=Y)
        net.add_node(name=name_r, x=X, y=Y)
        # Add consumers
        if "SimpleDistrict" in name:
            net.add_consumer(
                name=name,
                start_node=name_s,
                end_node=name_r,
                control_type="mass_flow",
                setpoint_type_hyd="mass_flow",
                setpoint_value_hyd=mdot_consumers,
                setpoint_value_hx=delta_t_consumers,
            )
        # Add producers
        if name == "i":
            net.add_producer(
                name="producer",
                start_node=name_r,
                end_node=name_s,
                setpoint_type_hx="t_out",
                setpoint_value_hx=t_supply_producer,
                setpoint_type_hyd="pressure",
                setpoint_value_hyd=-10000,
                static_pressure=100000,
            )

    # Add pipes
    for i in pipe_data.index:
        row = pipe_data.loc[i]
        start = row["start_node"]
        end = row["end_node"]
        name_s = f"{start}_to_{end}_s"
        name_r = f"{start}_to_{end}_r"
        start_s = f"{start}_s"
        end_s = f"{end}_s"
        start_r = f"{end}_r"
        end_r = f"{start}_r"
        length = row["length_m"]
        diameter = row["diameter_m"]
        thickness_ins = row["t_ins_m"]
        thickness_pipe = row["t_pipe_m"]

        net.add_pipe(
            name=name_s,
            start_node=start_s,
            end_node=end_s,
            diameter=diameter,
            length=length,
            roughness=roughness_pipe,
            k_insulation=lambda_ins,
            insulation_thickness=thickness_ins,
            k_internal_pipe=lambda_pipe,
            internal_pipe_thickness=thickness_pipe,
            casing_thickness=0.0,
            depth=1.5,
            line="supply",
        )
        net.add_pipe(
            name=name_r,
            start_node=start_r,
            end_node=end_r,
            diameter=diameter,
            length=length,
            roughness=roughness_pipe,
            k_insulation=lambda_ins,
            insulation_thickness=thickness_ins,
            k_internal_pipe=lambda_pipe,
            internal_pipe_thickness=thickness_pipe,
            casing_thickness=0.0,
            depth=1.5,
            line="return",
        )

    return net

# Simulation

In [57]:
def main():
    """
    Main function to run the simulation and save the results as a CSV file
    """
    # Create net, fluid and soil
    net = create_net()
    fluid = Fluid(name="Water", isconstant=True, cp=4180, mu=0.0005434, rho=988, k=0.64)
    soil = Soil(k=0.0, temp=10)

    # Run simulation
    hyd_kwargs = {"error_threshold": 1}
    thermal_kwargs = {"error_threshold": 1e-10}
    loop = SimpleStep(
        hydraulic_sim_kwargs=hyd_kwargs, thermal_sim_kwargs=thermal_kwargs
    )

    import time

    start_time = time.perf_counter()

    res = loop.execute(net=net, fluid=fluid, soil=soil)

    end_time = time.perf_counter()
    sim_time = end_time - start_time

    print(f"Average simulation time: {sim_time:.2f} s")


    # KPI CALCULATION
    # Mass flow
    mdot_supply = res["edges"]["mass_flow"][0, net.producers_mask[0]] * 3600

    # Pressure
    nodes, pressure = net.nodes(data="pressure")
    node_pressure = dict(zip(nodes, pressure))

    dp_i_to_e_s = node_pressure["i_s"] - node_pressure["e_s"]
    dp_a_to_i_r = node_pressure["a_r"] - node_pressure["i_r"]
    dp_i_to_h_r = node_pressure["i_r"] - node_pressure["h_r"]

    # Temperature
    nodes, temperature = net.nodes(data="temperature")
    node_temperature = dict(zip(nodes, temperature))
    node_names = ["i", "h", "g", "f", "e", "SimpleDistrict_1"]

    # Energy
    edges, energy = net.edges(data="delta_q")
    edges = [(u, v) for (u, v) in edges]
    q_w = dict(zip(edges, energy))
    i_to_h_q_w = q_w[("i_s", "h_s")]
    supply_q_w = q_w[("i_r", "i_s")]

    kpis = {
        "Mass flow rate supply i (kg/h)": mdot_supply,
        "Pressure drop supply between i and e (Pa)": abs(dp_i_to_e_s),
        "Pressure drop return between a and i (Pa)": abs(dp_a_to_i_r),
        "Pressure drop return between i and h (Pa)": abs(dp_i_to_h_r),
    }

    for l in ["s", "r"]:
        label = "supply" if l == "s" else "return"
        for n in node_names:
            node_key = f"{n}_{l}"
            kpis[f"Fluid temperature {label} {n} (°C)"] = node_temperature[node_key]

    kpis["Heat loss supply between i and h (W)"] = abs(i_to_h_q_w)
    kpis["Total heat load supplied by heat source (W)"] = abs(supply_q_w)

    results_df = pd.DataFrame(
        list(kpis.items()),
        columns=["KPI", "pydhn"]
    )

    return results_df

In [58]:
results_pydhn = main()

base_path = os.getcwd()
results_folder = os.path.join(base_path, 'Results')
results_file = os.path.join(results_folder, "network_CE0_pydhn_results.csv")
results_pydhn.to_csv(results_file, sep=";", index=False)

print(f"Results for CE0 implemented in pyDHN saved in: {results_file}")

Loop started
Hydraulic simulation converged after 0 iterations with an error of 3.637978807091713e-12 Pa
Thermal simulation converged after 1 iterations with an error of 2.842170943040401e-14 °C
Loop completed
Average simulation time: 0.02 s
Results for CE0 implemented in pyDHN saved in: C:\Users\sara.ferrero\PycharmProjects\OSMSES_2026_Ferrero\Results\network_CE0_pydhn_results.csv


C:\Users\sara.ferrero\PycharmProjects\PandaPipesModel\.venv\Lib\site-packages\pydhn\solving\loops.py:168: UserWarning: Running a thermal simulation without a ts_id specified. This can lead to unexpected behaviours in dynamic components. To suppress this warning, pass a ts_id value when calling .execute()
  warn(


In [59]:
results_pydhn

,KPI,pydhn
0,Mass flow rate supply i (kg/h),8848.000000
1,Pressure drop supply between i and e (Pa),23105.327359
2,Pressure drop return between a and i (Pa),23105.327359
3,Pressure drop return between i and h (Pa),5828.009424
4,Fluid temperature supply i (°C),70.000000
5,Fluid temperature supply h (°C),69.939137
6,Fluid temperature supply g (°C),69.869487
7,Fluid temperature supply f (°C),69.762907
8,Fluid temperature supply e (°C),69.593664
9,Fluid temperature supply SimpleDistrict_1 (°C),69.457338
